# Домашнее задание 1_1

In [1]:
!pwd

/home/jovyan/work/hw_1_1


In [2]:
!ls -la

total 32
drwxrwxrwx 1 root root  4096 Sep 20 20:02 .
drwxrwxrwx 1 root root  4096 Sep 20 17:20 ..
drwxr-xr-x 1 root root  4096 Sep 20 17:24 data
-rwxrwxrwx 1 root root 22992 Sep 20 20:02 hw_1_1.ipynb
drwxr-xr-x 1 root root  4096 Sep 20 17:35 .ipynb_checkpoints
-rw-r--r-- 1 root root    14 Sep 20 18:17 task1_count.txt
-rw-r--r-- 1 root root   205 Sep 20 18:15 task1_mapper.py
-rw-r--r-- 1 root root   219 Sep 20 18:17 task1_reducer.py
-rw-r--r-- 1 root root   244 Sep 20 19:23 task2_mapper
-rw-r--r-- 1 root root   248 Sep 20 19:54 task2_mapper.py
-rw-r--r-- 1 root root  1033 Sep 20 19:43 task2_reducer
-rw-r--r-- 1 root root   906 Sep 20 19:57 task2_reducer.py


In [3]:
!hdfs dfs -ls /

Found 2 items
drwxr-xr-x   - hadoop users          0 2026-09-20 20:00 /hw_1_1
drwxr-xr-x   - hadoop users          0 2026-09-20 18:29 /tmp


In [4]:
!head -n 3 data/bible.txt

In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. 
And God said, Let there be light: and there was light. 
And God saw the light, that it was good: and God divided the light from the darkness. 


## Задание 1

#### Шаг Map

In [5]:
%%writefile task1_mapper.py

import sys


def mapper():
    for line in sys.stdin:
        for word in line.strip().split():
            if len(word) > 4:
                print(f"{word}\t1")


if __name__ == "__main__":
    mapper()

Overwriting task1_mapper.py


#### Шаг Reduce

In [6]:
%%writefile task1_reducer.py

import sys


def reducer():
    count = 0

    for line in sys.stdin:
        _word, cnt = line.strip().split('\t', 1)
        count += int(cnt)

    print(f"Count: {count}")


if __name__ == "__main__":
    reducer()

Overwriting task1_reducer.py


#### bash команда

In [7]:
!ls

data		 task1_mapper.py   task2_mapper.py
hw_1_1.ipynb	 task1_reducer.py  task2_reducer
task1_count.txt  task2_mapper	   task2_reducer.py


In [8]:
%%time
! cat data/bible.txt | python task1_mapper.py | python task1_reducer.py > task1_count.txt

CPU times: user 5.77 ms, sys: 5.17 ms, total: 10.9 ms
Wall time: 345 ms


In [9]:
cat task1_count.txt

Count: 266384


#### MR таска

In [10]:
!hdfs dfs -mkdir -p /hw_1_1/task1/input

In [11]:
!hdfs dfs -put -f data/bible.txt /hw_1_1/task1/input

In [12]:
!hdfs dfs -ls /hw_1_1/task1/input

Found 1 items
-rw-r--r--   1 hadoop users    4047392 2026-09-20 20:03 /hw_1_1/task1/input/bible.txt


In [13]:
!hdfs dfs -rm -r -f /hw_1_1/task1/output

In [14]:
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
  -D mapreduce.job.name="hw_1_1_task1_total_words" \
  -D mapreduce.job.reduces=1 \
  -files task1_mapper.py,task1_reducer.py \
  -mapper "python3 task1_mapper.py" \
  -reducer "python3 task1_reducer.py" \
  -input /hw_1_1/task1/input \
  -output /hw_1_1/task1/output

packageJobJar: [/tmp/hadoop-unjar3006584038820333541/] [] /tmp/streamjob6884409848819911858.jar tmpDir=null
2026-09-20 20:03:45,853 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.18.0.3:8032
2026-09-20 20:03:46,167 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.18.0.3:8032
2026-09-20 20:03:46,455 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1789924421728_0003
2026-09-20 20:03:47,153 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-09-20 20:03:47,661 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-09-20 2

In [15]:
!hdfs dfs -ls /hw_1_1/task1/output

Found 2 items
-rw-r--r--   1 hadoop users          0 2026-09-20 20:04 /hw_1_1/task1/output/_SUCCESS
-rw-r--r--   1 hadoop users         15 2026-09-20 20:04 /hw_1_1/task1/output/part-00000


In [16]:
!hdfs dfs -rm /hw_1_1/task1/output/_SUCCESS

Deleted /hw_1_1/task1/output/_SUCCESS


In [17]:
!hdfs dfs -cat /hw_1_1/task1/output/part-*

Count: 266384	


## Задание 2

In [18]:
%%writefile task2_mapper.py

import sys
import csv


def mapper():
    reader = csv.reader(sys.stdin, delimiter=';')
    for row in reader:
        link, date = row
        date = date.split()[0]
        print(f"{date}\x1f{link}\t1")


if __name__ == "__main__":
    mapper()

Overwriting task2_mapper.py


In [19]:
%%writefile task2_reducer.py

import sys
import heapq


def reducer():
    current_date = ""
    current_site = ""
    current_cnt = 0
    top5 = []

    for line in sys.stdin:
        text, cnt = line.strip().split('\t', 1)
        date, site = text.split('\x1f')

        if date != current_date:
            print(current_date)
            for count, link in sorted(top5, key=lambda x: -x[0]):
                print(link, "---", count)
            print("===")
            current_date = date
            top5 = []
        elif site != current_site:
            item = (current_cnt, current_site)
            if len(top5) < 5:
                heapq.heappush(top5, item)
            elif current_cnt > top5[0][0]:
                heapq.heapreplace(top5, item)

            current_site = site
            current_cnt = int(cnt)
        else:
            current_cnt += int(cnt)


if __name__ == "__main__":
    reducer()            

Overwriting task2_reducer.py


In [20]:
!ls

data		 task1_mapper.py   task2_mapper.py
hw_1_1.ipynb	 task1_reducer.py  task2_reducer
task1_count.txt  task2_mapper	   task2_reducer.py


In [21]:
!cat data/Poseschenia_sai_774_tov.csv | python3 task2_mapper.py | sort | python3 task2_reducer.py


===
2024-05-26
https://gonzales-bautista.com/ --- 335
http://smith.com/ --- 235
https://www.smith.com/ --- 221
http://www.smith.com/ --- 212
https://smith.com/ --- 212
===
2024-05-27
https://gonzales-bautista.com/ --- 376
https://www.smith.com/ --- 270
https://smith.com/ --- 236
http://smith.com/ --- 215
http://www.smith.com/ --- 208
===
2024-05-28
https://gonzales-bautista.com/ --- 368
https://smith.com/ --- 256
https://www.smith.com/ --- 251
http://smith.com/ --- 224
http://www.smith.com/ --- 204
===
2024-05-29
https://gonzales-bautista.com/ --- 402
https://www.smith.com/ --- 242
http://www.smith.com/ --- 223
https://smith.com/ --- 220
http://smith.com/ --- 206
===
2024-05-30
https://gonzales-bautista.com/ --- 353
https://smith.com/ --- 246
https://www.smith.com/ --- 239
http://smith.com/ --- 229
http://www.smith.com/ --- 225
===
2024-05-31
https://gonzales-bautista.com/ --- 374
https://www.smith.com/ --- 244
http://smith.com/ --- 228
http://www.smith.com/ --- 221
https://smith.com/

In [22]:
!hdfs dfs -mkdir -p /hw_1_1/task2/input

In [23]:
!hdfs dfs -put -f data/Poseschenia_sai_774_tov.csv /hw_1_1/task2/input/

In [24]:
!hdfs dfs -ls /hw_1_1/task2/input

Found 1 items
-rw-r--r--   1 hadoop users   36443383 2026-09-20 20:04 /hw_1_1/task2/input/Poseschenia_sai_774_tov.csv


In [25]:
!hdfs dfs -rm -r -f /hw_1_1/task2/output

Deleted /hw_1_1/task2/output


In [26]:
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
  -D mapreduce.job.name="hw_1_1_task2_top5_sites" \
  -D mapreduce.job.reduces=1 \
  -files task2_mapper.py,task2_reducer.py \
  -mapper "python3 task2_mapper.py" \
  -reducer "python3 task2_reducer.py" \
  -input /hw_1_1/task2/input \
  -output /hw_1_1/task2/output

packageJobJar: [/tmp/hadoop-unjar3017697623660928052/] [] /tmp/streamjob380524413581493761.jar tmpDir=null
2026-09-20 20:04:29,306 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.18.0.3:8032
2026-09-20 20:04:29,638 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.18.0.3:8032
2026-09-20 20:04:29,962 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1789924421728_0004
2026-09-20 20:04:30,656 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-09-20 20:04:30,733 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-09-20 20

In [27]:
!hdfs dfs -ls /hw_1_1/task2/output

Found 2 items
-rw-r--r--   1 hadoop users          0 2026-09-20 20:04 /hw_1_1/task2/output/_SUCCESS
-rw-r--r--   1 hadoop users       1231 2026-09-20 20:04 /hw_1_1/task2/output/part-00000


In [28]:
!hdfs dfs -cat /hw_1_1/task2/output/part-*

	
===	
2024-05-26	
https://gonzales-bautista.com/ --- 335	
http://smith.com/ --- 235	
https://www.smith.com/ --- 221	
http://www.smith.com/ --- 212	
https://smith.com/ --- 212	
===	
2024-05-27	
https://gonzales-bautista.com/ --- 376	
https://www.smith.com/ --- 270	
https://smith.com/ --- 236	
http://smith.com/ --- 215	
http://www.smith.com/ --- 208	
===	
2024-05-28	
https://gonzales-bautista.com/ --- 368	
https://smith.com/ --- 256	
https://www.smith.com/ --- 251	
http://smith.com/ --- 224	
http://www.smith.com/ --- 204	
===	
2024-05-29	
https://gonzales-bautista.com/ --- 402	
https://www.smith.com/ --- 242	
http://www.smith.com/ --- 223	
https://smith.com/ --- 220	
http://smith.com/ --- 206	
===	
2024-05-30	
https://gonzales-bautista.com/ --- 353	
https://smith.com/ --- 246	
https://www.smith.com/ --- 239	
http://smith.com/ --- 229	
http://www.smith.com/ --- 225	
===	
2024-05-31	
https://gonzales-bautista.com/ --- 374	
https://www.smith.com/ --- 244	
http://smith.com/ --- 228	
http://